[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Your First Test


## What you will be able to do

Write tests in a file and run them with pytest. Read its report, from the row of dots to the values
behind a failed `assert`, choose which tests run by file, by name or by keyword, stop at the first
failure or run only the failures again, and know how pytest finds a test, so that none is skipped
without your knowing.


## The idea

### The problem

The **Why Test** notebook ran its tests with `run_tests`, a few lines that found functions by name
and reported the ones that raised. It did the job inside a notebook, and a project needs more than
it gave. The tests belong in files beside the code, where anybody can run them, not in a notebook's
memory. Every run should start a new Python, so that a module changed on disk is never tested as the
copy imported an hour earlier. A failed `assert` should say what the values were, where `run_tests`
said only `AssertionError()`. And a suite of five hundred tests needs a way to run one of them, or
only the ones that failed last time.

Writing that runner well is a project of its own, and every Python project would need one. pytest is
that runner, written once.

### What pytest is

> **pytest** is a test runner. Run with no arguments, it **collects** tests from the current folder
> and the folders below it: in every file whose name matches `test_*.py` or `*_test.py`, it takes
> each function whose name starts with `test`, and each such method of a class whose name starts
> with `Test`. It runs every test it collected. A test **fails** when it raises, whether from an
> `assert` or from anything else. The report shows a character for each test, the details of every
> failure, and a line of counts, and the command ends with an **exit code**: 0 when every test
> passed, and 1 when any test failed.

### Why it works that way

- **A test is found by its name.** A function named `test_...` in a file named `test_...` is in the
  suite, with no list to keep up to date. A function or a file with any other name is not, and
  nothing warns you, which is why `--collect-only`, which lists what pytest found, is worth knowing.
- **A plain `assert` shows its values.** pytest rewrites the `assert` statements in a test file as it
  imports the file, so a failure reports `assert 57.6 == 32` and where `57.6` came from, where
  Python's own `AssertionError` carries no message at all.
- **Every run is a new Python.** `python -m pytest` starts a program, which imports every file as it
  is on disk.
- **Every failure is reported.** A failing test is reported and the rest still run, so one report
  shows every failure, each with the details that explain it.
- **The report is for you, and the exit code is for programs.** A person reads the dots and the
  failures. A script, or a server that runs the tests on every push, reads the number the command
  ends with.

### Where you will meet this

pytest is the runner most Python projects use. NumPy and pandas run their suites with it, as the
**Why Test** notebook said, and GitHub's guide to building and testing Python runs `pytest` in its
example workflow, the kind of file the **Continuous Integration** notebook writes. pytest also runs
tests written for `unittest`, the test framework in Python's standard library, so an older suite runs
under it unchanged. Colab has pytest installed. On your own computer, you install it into a project's
virtual environment with `python -m pip install pytest`, as the **Virtual Environments** notebook
shows.

### What this notebook covers

- Tests in a file of their own, and the command that runs them
- The report: a character for each test, the counts, and `-v` for a line per test
- A failure, read line by line
- The exit code, which tells a program how the run went
- Stopping at the first failure with `-x`, and running only the failures again with `--lf`
- What a plain `assert` shows for dictionaries and lists, and an `assert` with a message
- How pytest finds tests, and `--collect-only`
- Running part of a suite: one file, one test, or the tests `-k` matches
- A bug fixed with the suite, from the failing test to the passing run
- Six errors, from a test file pytest never reads to floats compared with `==`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import subprocess
import sys
from pathlib import Path

Path("readings.py").write_text('''
def to_fahrenheit(celsius):
    return (celsius + 32) * 9 / 5
''')

Path("test_readings.py").write_text('''from readings import to_fahrenheit


def test_freezing_point():
    assert to_fahrenheit(0) == 32
''')

finished = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(finished.stdout, end="")
```

```
F                                                                        [100%]
=================================== FAILURES ===================================
_____________________________ test_freezing_point ______________________________

    def test_freezing_point():
>       assert to_fahrenheit(0) == 32
E       assert 57.6 == 32
E        +  where 57.6 = to_fahrenheit(0)

test_readings.py:5: AssertionError
=========================== short test summary info ============================
FAILED test_readings.py::test_freezing_point - assert 57.6 == 32
1 failed in 0.01s
```

One test, a formula with its brackets in the wrong place, and pytest's report: the test that failed,
the line that failed, the values on both sides of `==`, and where `57.6` came from. Nothing called
the test: pytest found it by its name. The rest of the notebook runs pytest on a project and reads
every part of that report.


## Setup

Seven imports, and a folder for the project.

- `version`, from `importlib.metadata`, reads the version of pytest this notebook runs
- `subprocess` runs pytest as a program of its own, as a terminal would
- `sys` names the Python that runs it
- `os` passes pytest the environment, and sets two variables in it: `NO_COLOR`, which asks pytest to
  print plain text, and `PYTHONDONTWRITEBYTECODE`, which stops Python from saving compiled copies of
  files in `__pycache__`, since a notebook can rewrite a file so quickly that a saved copy still
  looks up to date
- `re` takes out of pytest's report the parts that differ between computers
- `Path` makes the project's folder, and reads what pytest leaves in it
- `shutil` removes the scratch folder at the end

The project goes in `scratch/stations`. Colab has pytest installed, and this notebook runs version
8.4.2. If the cell reports no pytest, run `!pip install pytest==8.4.2` in a cell of its own, and run
this cell again.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

PROJECT = Path("scratch/stations")
PROJECT.mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun

print("pytest", version("pytest"), "| the project's folder:", PROJECT)


pytest 8.4.2 | the project's folder: scratch/stations


## Worked examples

### Tests in a file

A project keeps its tests in files of their own. The module is the one the **Why Test** notebook
finished with:


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings."""
    by_station = {}
    for line in lines:
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


The tests are that notebook's tests, moved into `test_readings.py`. The file imports what it tests
from the module, as any code that uses the module does:


In [3]:
%%writefile scratch/stations/test_readings.py
from readings import mean, parse_reading, to_fahrenheit


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_mean_skips_missing_readings():
    assert mean([4.2, None, 5.8]) == 5.0


def test_mean_of_no_readings_is_none():
    assert mean([None, None]) is None


def test_an_empty_reading_is_none():
    assert parse_reading("Svalbard,") == ("Svalbard", None)


def test_freezing_point_in_fahrenheit():
    assert to_fahrenheit(0) == 32


def test_boiling_point_in_fahrenheit():
    assert to_fahrenheit(100) == 212


Writing scratch/stations/test_readings.py


Nothing in the file calls a test. The tests are functions waiting to be found, and finding them is
pytest's job.

### Running pytest

In a terminal, you run pytest from the project's folder:

```
cd stations
python -m pytest
```

`python -m pytest` runs the pytest installed for that `python`. Typing `pytest` alone runs whichever
pytest the shell finds first, and it also leaves the current folder off the import path, which the
**Project Layout** notebook comes back to. A notebook has no terminal, so `run_pytest` runs the same
command in the project's folder with `subprocess.run` and prints the report. `pytest_report` does the
running and returns the report, for a cell that prints only part of it:


In [4]:
def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, folder=PROJECT):
    """Run python -m pytest in the folder, as a terminal would, and print its report."""
    print(pytest_report(*arguments, folder=folder))


run_pytest()


============================= test session starts ==============================
collected 6 items

test_readings.py ......                                                  [100%]

============================== 6 passed ===============================


That is pytest's report, with a few things held still that would otherwise make this notebook's
output differ from what you see:

- `--no-header` leaves out the lines about the computer: its operating system, its Python, the
  folder, and the plugins installed beside pytest
- `PYTEST_DISABLE_PLUGIN_AUTOLOAD` stops those plugins from loading, since Colab has several and
  every computer has its own
- `PYTHONNODEBUGRANGES` stops Python from recording where each expression sits in a line, which
  pytest uses to underline part of a failing line with `^^^`, and which can differ between Python
  versions
- `COLUMNS` makes the report's lines 80 characters wide, whatever the width of the window
- the folder's path and the path to pytest's own files are taken out of the paths pytest prints, and
  so is the time the run took, such as ` in 0.02s`, which changes on every run

In a terminal, you see all of those.

### Reading the report

`collected 6 items` counts the tests pytest found. The next line names the file and gives a character
for each test in it, in the order they ran: `.` for a test that passed, `F` for one that failed, and
`E` for an error outside the test itself, such as a file that cannot be imported. The percentage is
how far through the suite the run had got. The last line gives the counts.

`-v`, for verbose, prints a line for each test instead of a character. Each line starts with the
test's **node ID**, the file and the test's name joined by `::`, which is how pytest names a test
everywhere, from this report to the command that runs a single test:


In [5]:
run_pytest("-v")


============================= test session starts ==============================
collecting ... collected 6 items

test_readings.py::test_mean_of_two_readings PASSED                       [ 16%]
test_readings.py::test_mean_skips_missing_readings PASSED                [ 33%]
test_readings.py::test_mean_of_no_readings_is_none PASSED                [ 50%]
test_readings.py::test_an_empty_reading_is_none PASSED                   [ 66%]
test_readings.py::test_freezing_point_in_fahrenheit PASSED               [ 83%]
test_readings.py::test_boiling_point_in_fahrenheit PASSED                [100%]

============================== 6 passed ===============================


### A failure, line by line

Now a bug. The brackets in `to_fahrenheit` move, so that it adds 32 before it multiplies:


In [6]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings."""
    by_station = {}
    for line in lines:
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return (celsius + 32) * 9 / 5


Overwriting scratch/stations/readings.py


In [7]:
run_pytest()


============================= test session starts ==============================
collected 6 items

test_readings.py ....FF                                                  [100%]

=================================== FAILURES ===================================
______________________ test_freezing_point_in_fahrenheit _______________________

    def test_freezing_point_in_fahrenheit():
>       assert to_fahrenheit(0) == 32
E       assert 57.6 == 32
E        +  where 57.6 = to_fahrenheit(0)

test_readings.py:21: AssertionError
_______________________ test_boiling_point_in_fahrenheit _______________________

    def test_boiling_point_in_fahrenheit():
>       assert to_fahrenheit(100) == 212
E       assert 237.6 == 212
E        +  where 237.6 = to_fahrenheit(100)

test_readings.py:25: AssertionError
=========================== short test summary info ============================
FAILED test_readings.py::test_freezing_point_in_fahrenheit - assert 57.6 == 32
FAILED test_readings.py::test_b

The run needed no reload: pytest started a new Python, which imported the module as the file is now.
Here is the report, part by part:

| In the report | What it says |
|---|---|
| `....FF` | the tests in the order they ran: four passed, and the last two failed |
| `FAILURES`, then a heading for each failed test | the details of each failure, under the test's name |
| the test's code, with `>` beside one line | the line that raised |
| `E       assert 57.6 == 32` | the `assert`, with the value of each side in place of its expression |
| `E        +  where 57.6 = to_fahrenheit(0)` | where a value came from: the call that returned it |
| `test_readings.py:21: AssertionError` | the file, the line number, and the exception the line raised |
| `short test summary info` | one line for each failure: its node ID and the start of its message |
| `2 failed, 4 passed` | the counts |

The values do most of the work. `to_fahrenheit(0)` returned `57.6`, and 57.6 is (0 + 32) × 9 / 5,
which points at the order of the arithmetic before anybody opens the module.

### The exit code, for programs

A command ends with a number, its exit code, which a terminal does not show and a program reads.
`exit_code` runs pytest with some arguments and returns that number, without printing the report:


In [8]:
def exit_code(*arguments):
    """The exit code of python -m pytest, run in the project's folder with these arguments."""
    return subprocess.run([sys.executable, "-m", "pytest", *arguments], cwd=PROJECT, capture_output=True).returncode


for arguments in [(), ("-k", "mean"), ("-k", "no_such_test"), ("--no-such-option",)]:
    print(f"python -m pytest {' '.join(arguments):<20} exit code {exit_code(*arguments)}")


python -m pytest                      exit code 1
python -m pytest -k mean              exit code 0
python -m pytest -k no_such_test      exit code 5
python -m pytest --no-such-option     exit code 4


The whole suite has two failures, so 1. `-k mean` runs only the tests with `mean` in their names,
which all pass, so 0. A run that collects nothing ends with 5, and a mistake in the command itself
with 4. pytest's documentation lists them:

| Exit code | Means |
|---|---|
| 0 | every test that was collected ran and passed |
| 1 | tests ran, and at least one failed |
| 2 | the run was interrupted, for instance by a test file that could not be imported |
| 3 | pytest itself failed |
| 4 | the command was wrong, such as an option pytest does not have |
| 5 | no tests were collected |

A server that runs the tests on every push decides with this number whether the push passed, which
the **Continuous Integration** notebook shows.

### Stopping at the first failure, and running only the failures again

`-x` stops the run at the first failure, which saves time when one broken thing is likely to break
many tests:


In [9]:
run_pytest("-x")


============================= test session starts ==============================
collected 6 items

test_readings.py ....F

=================================== FAILURES ===================================
______________________ test_freezing_point_in_fahrenheit _______________________

    def test_freezing_point_in_fahrenheit():
>       assert to_fahrenheit(0) == 32
E       assert 57.6 == 32
E        +  where 57.6 = to_fahrenheit(0)

test_readings.py:21: AssertionError
=========================== short test summary info ============================
FAILED test_readings.py::test_freezing_point_in_fahrenheit - assert 57.6 == 32
!!!!!!!!!!!!!!!!!!!!!!!!!! stopping after 1 failures !!!!!!!!!!!!!!!!!!!!!!!!!!!
========================= 1 failed, 4 passed ==========================


The run stopped after the first failed test, and `test_boiling_point_in_fahrenheit` never ran. Now
the fix, with the brackets back where they were:


In [10]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings."""
    by_station = {}
    for line in lines:
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Overwriting scratch/stations/readings.py


`--lf`, for last failed, runs only the tests that failed the last time they ran, which is the
quickest way to see whether a fix worked:


In [11]:
run_pytest("--lf")


============================= test session starts ==============================
collected 2 items
run-last-failure: rerun previous 2 failures

test_readings.py ..                                                      [100%]

============================== 2 passed ===============================


Both failures pass now, and the full suite confirms that the fix broke nothing else:


In [12]:
run_pytest("-q")


......                                                                   [100%]
6 passed


`-q`, for quiet, is the opposite of `-v`: no heading, and the counts on a line of their own. pytest
remembered which tests failed in a folder it made, `.pytest_cache`, beside the tests:


In [13]:
print(sorted(path.name for path in PROJECT.iterdir()))
print(sorted(path.name for path in (PROJECT / ".pytest_cache").iterdir()))


['.pytest_cache', 'readings.py', 'test_readings.py']
['.gitignore', 'CACHEDIR.TAG', 'README.md', 'v']


The cache holds a `.gitignore` of its own, so Git leaves the folder out of a repository without being
told.

### What a plain assert shows

A failure means the code and a test disagree, and either one can be wrong. `test_summary.py` has two
tests whose expectations are wrong on purpose, to show what pytest prints when an `assert` compares
two dictionaries, and when an `assert` carries a message after its condition:


In [14]:
%%writefile scratch/stations/test_summary.py
from readings import summarize

TUESDAY = ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
           "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]


def test_summary_of_tuesday():
    assert summarize(TUESDAY) == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.5}


def test_every_station_is_in_the_summary():
    stations = sorted(summarize(TUESDAY))
    assert stations == ["Alta", "Bergen", "Oslo", "Svalbard", "Tromso"], "a station is missing from the summary"


Writing scratch/stations/test_summary.py


In [15]:
run_pytest("test_summary.py")


============================= test session starts ==============================
collected 2 items

test_summary.py FF                                                       [100%]

=================================== FAILURES ===================================
___________________________ test_summary_of_tuesday ____________________________

    def test_summary_of_tuesday():
>       assert summarize(TUESDAY) == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.5}
E       AssertionError: assert {'Bergen': 5....Tromso': -6.0} == {'Bergen': 5....Tromso': -6.5}
E         
E         Omitting 3 identical items, use -vv to show
E         Differing items:
E         {'Tromso': -6.0} != {'Tromso': -6.5}
E         Use -v to get more diff

test_summary.py:8: AssertionError
_____________________ test_every_station_is_in_the_summary _____________________

    def test_every_station_is_in_the_summary():
        stations = sorted(summarize(TUESDAY))
>       assert stations == ["Alta", "Ber

For two dictionaries, pytest lists the items that differ, `{'Tromso': -6.0} != {'Tromso': -6.5}`,
and says how many items were the same; `-vv` shows them all. For two lists, it names the first place
they differ. The message after the comma comes first, after `AssertionError:`, with the comparison
under it.

This time the tests were wrong, not the code: Tromso's mean is -6.0, and Alta has no station. The
corrected file also moves Tuesday's lines into a function, `tuesday`, which both tests call:


In [16]:
%%writefile scratch/stations/test_summary.py
from readings import summarize


def tuesday():
    """Tuesday's lines of readings, for the tests below."""
    return ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
            "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]


def test_summary_of_tuesday():
    assert summarize(tuesday()) == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.0}


def test_every_station_is_in_the_summary():
    stations = sorted(summarize(tuesday()))
    assert stations == ["Bergen", "Oslo", "Svalbard", "Tromso"], "a station is missing from the summary"


Overwriting scratch/stations/test_summary.py


In [17]:
run_pytest("test_summary.py")


============================= test session starts ==============================
collected 2 items

test_summary.py ..                                                       [100%]

============================== 2 passed ===============================


### How pytest finds tests

pytest decides what is a test from names alone:

| pytest collects | For example | And leaves alone |
|---|---|---|
| files named `test_*.py` or `*_test.py` | `test_readings.py` | `readings.py`, `check_readings.py` |
| functions whose names start with `test`, outside a class | `def test_summary_of_tuesday():` | `def tuesday():` |
| methods whose names start with `test`, in a class whose name starts with `Test` and that has no `__init__` | `def test_empty(self):` in `class TestMean:` | the same class with an `__init__` |

`--collect-only` lists what pytest would run, without running it, and `-q` gives just the node IDs:


In [18]:
run_pytest("--collect-only", "-q")


test_readings.py::test_mean_of_two_readings
test_readings.py::test_mean_skips_missing_readings
test_readings.py::test_mean_of_no_readings_is_none
test_readings.py::test_an_empty_reading_is_none
test_readings.py::test_freezing_point_in_fahrenheit
test_readings.py::test_boiling_point_in_fahrenheit
test_summary.py::test_summary_of_tuesday
test_summary.py::test_every_station_is_in_the_summary

8 tests collected


Eight tests, from two files. `tuesday` is not among them, since its name does not start with `test`,
and neither is anything in `readings.py`. A class is how a file groups tests, which the **Test
Structure** notebook takes up; its tests have node IDs such as
`test_readings.py::TestMean::test_empty`.

### Running part of a suite

A suite grows, and while you work on one part you run that part. A path runs one file, a node ID runs
one test, and `-k` runs the tests whose names match an expression:


In [19]:
run_pytest("test_summary.py", "-q")
print()
run_pytest("test_readings.py::test_mean_of_no_readings_is_none", "-v")
print()
run_pytest("-k", "fahrenheit or empty", "-v")


..                                                                       [100%]
2 passed

============================= test session starts ==============================
collecting ... collected 1 item

test_readings.py::test_mean_of_no_readings_is_none PASSED                [100%]

============================== 1 passed ===============================

============================= test session starts ==============================
collecting ... collected 8 items / 5 deselected / 3 selected

test_readings.py::test_an_empty_reading_is_none PASSED                   [ 33%]
test_readings.py::test_freezing_point_in_fahrenheit PASSED               [ 66%]
test_readings.py::test_boiling_point_in_fahrenheit PASSED                [100%]

======================= 3 passed, 5 deselected ========================


`-k` matches a word against the names of the tests and the classes they are in, ignoring case, and
joins words with `and`, `or` and `not`. The deselected tests were collected and not run, and the
counts say so.

### A bug fixed with the suite

The pieces of this notebook, in one fix. A file of readings arrives with a blank line at its end,
and the summary stops. First, the test that shows it, added to `test_summary.py`:


In [20]:
%%writefile scratch/stations/test_summary.py
from readings import summarize


def tuesday():
    """Tuesday's lines of readings, for the tests below."""
    return ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
            "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]


def test_summary_of_tuesday():
    assert summarize(tuesday()) == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.0}


def test_every_station_is_in_the_summary():
    stations = sorted(summarize(tuesday()))
    assert stations == ["Bergen", "Oslo", "Svalbard", "Tromso"], "a station is missing from the summary"


def test_a_blank_line_is_skipped():
    assert summarize(["Bergen,4.2", "", "Bergen,5.8"]) == {"Bergen": 5.0}


Overwriting scratch/stations/test_summary.py


In [21]:
run_pytest("-k", "blank")


============================= test session starts ==============================
collected 9 items / 8 deselected / 1 selected

test_summary.py F                                                        [100%]

=================================== FAILURES ===================================
_________________________ test_a_blank_line_is_skipped _________________________

    def test_a_blank_line_is_skipped():
>       assert summarize(["Bergen,4.2", "", "Bergen,5.8"]) == {"Bergen": 5.0}

test_summary.py:20: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
readings.py:26: in summarize
    station, celsius = parse_reading(line)
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

line = ''

    def parse_reading(line):
        """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.
    
        A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
        """
>  

This failure is not an `assert` that was false. The test called `summarize`, which called
`parse_reading`, and the report follows the calls down to the line that raised: the test's line, the
line in `summarize` that made the call, and `parse_reading` with the argument it was given,
`line = ''`. An empty line has no comma, so `split(",")` returned one piece where the unpacking
needed two. The fix belongs in `summarize`, which skips a line that holds nothing but whitespace:


In [22]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if not line.strip():
            continue
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Overwriting scratch/stations/readings.py


In [23]:
run_pytest("--lf")
print()
run_pytest("-q")
print("exit code:", exit_code())


============================= test session starts ==============================
collected 1 item
run-last-failure: rerun previous 1 failure (skipped 1 file)

test_summary.py .                                                        [100%]

============================== 1 passed ===============================

.........                                                                [100%]
9 passed
exit code: 0


### Where each part came from

| In the fix | What it relies on | The section that showed it |
|---|---|---|
| the test written first, in a file named `test_...` | tests found by their names | How pytest finds tests |
| `-k blank` | a keyword that picks tests by name | Running part of a suite |
| the report of the failure, down to `line = ''` | a failure's details, read line by line | A failure, line by line |
| a new Python for the run after the fix | no reload, since pytest imports the files as they are | A failure, line by line |
| `--lf` | pytest's memory of the tests that failed | Stopping at the first failure, and running only the failures again |
| the whole suite, and exit code 0 | every other test still passing, in a number a program can read | The exit code, for programs |

The fix took one test, one change to the module, and two runs: `--lf` to see the failing test pass,
and the whole suite to see that nothing else broke.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/02-your-first-test-solutions.ipynb).

**1.** Write `test_tasks.py` in the project's folder, with a test that checks that the mean of the
single reading `-6.3` is `-6.3`. Run pytest on that file alone.


In [24]:
# your code here


**2.** Add a second test to `test_tasks.py` that fails on purpose: the mean of `4.2` and `5.8` is
`5.8`. Run the file again, and find in the report the value the mean really had.


In [25]:
# your code here


**3.** Run only the failing test from task 2, by its node ID, with `-v`.


In [26]:
# your code here


**4.** Run every test in the project whose name contains `fahrenheit`, with `-v`.


In [27]:
# your code here


**5.** Print the exit code of pytest run on `test_tasks.py`. Then correct the failing test so that it
expects `5.0`, and print the exit code again.


In [28]:
# your code here


**6.** List every test in the project with `--collect-only -q`, and print how many tests there are,
counting the lines of the list that contain `::`.


In [29]:
# your code here


## Common errors

### No error, and a test file pytest never reads: a file named check_parsing.py


In [30]:
%%writefile scratch/stations/check_parsing.py
from readings import parse_reading


def test_a_reading_with_a_space_after_the_comma():
    assert parse_reading("Tromso, -6.3") == ("Tromso", -6.3)


Writing scratch/stations/check_parsing.py


In [31]:
run_pytest("-q")


.........                                                                [100%]
9 passed


The run passed with the same count as before, and the new test was not in it: `check_parsing.py`
does not match `test_*.py`, so pytest never opened it. Nothing says a file was skipped, because to
pytest it is not a test file. Name the file for pytest:


In [32]:
(PROJECT / "check_parsing.py").rename(PROJECT / "test_parsing.py")

run_pytest("-q")


..........                                                               [100%]
10 passed


### No error, and a test pytest never runs: a function named check_...


In [33]:
%%writefile scratch/stations/test_parsing.py
from readings import parse_reading


def test_a_reading_with_a_space_after_the_comma():
    assert parse_reading("Tromso, -6.3") == ("Tromso", -6.3)


def check_a_line_with_spaces_around_it():
    assert parse_reading("  Bergen,4.2  ") == ("Bergen", 4.2)


Overwriting scratch/stations/test_parsing.py


In [34]:
run_pytest("test_parsing.py", "-v")


============================= test session starts ==============================
collecting ... collected 1 item

test_parsing.py::test_a_reading_with_a_space_after_the_comma PASSED      [100%]

============================== 1 passed ===============================


The file is collected, and only one of its two tests ran: `check_a_line_with_spaces_around_it` is a
function pytest leaves alone, since its name does not start with `test`. `--collect-only` would have
listed one test for the file. Start the name with `test`:


In [35]:
source = (PROJECT / "test_parsing.py").read_text()
(PROJECT / "test_parsing.py").write_text(source.replace("def check_", "def test_"))

run_pytest("test_parsing.py", "-v")


============================= test session starts ==============================
collecting ... collected 2 items

test_parsing.py::test_a_reading_with_a_space_after_the_comma PASSED      [ 50%]
test_parsing.py::test_a_line_with_spaces_around_it PASSED                [100%]

============================== 2 passed ===============================


### PytestAssertRewriteWarning: assertion is always true, perhaps remove parentheses?


In [36]:
%%writefile scratch/stations/test_tuple.py
from readings import mean


def test_mean_of_two_readings():
    assert (mean([4.2, 5.8]) == 6.0, "the mean of 4.2 and 5.8")


Writing scratch/stations/test_tuple.py


In [37]:
run_pytest("test_tuple.py")


============================= test session starts ==============================
collected 1 item

test_tuple.py .                                                          [100%]

=============================== warnings summary ===============================
test_tuple.py:5
  test_tuple.py:5: PytestAssertRewriteWarning: assertion is always true, perhaps remove parentheses?
    assert (mean([4.2, 5.8]) == 6.0, "the mean of 4.2 and 5.8")

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
========================= 1 passed, 1 warning =========================


The test passed, although the mean of 4.2 and 5.8 is not 6.0. The parentheses made the condition and
the message into one tuple, and `assert` tested the tuple, which is true because it is not empty.
pytest warned, and the test still counts as passed. Without the parentheses, the message is a message
again, and the test fails as it should:


In [38]:
%%writefile scratch/stations/test_tuple.py
from readings import mean


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 6.0, "the mean of 4.2 and 5.8"


Overwriting scratch/stations/test_tuple.py


In [39]:
run_pytest("test_tuple.py")


============================= test session starts ==============================
collected 1 item

test_tuple.py F                                                          [100%]

=================================== FAILURES ===================================
__________________________ test_mean_of_two_readings ___________________________

    def test_mean_of_two_readings():
>       assert mean([4.2, 5.8]) == 6.0, "the mean of 4.2 and 5.8"
E       AssertionError: the mean of 4.2 and 5.8
E       assert 5.0 == 6.0
E        +  where 5.0 = mean([4.2, 5.8])

test_tuple.py:5: AssertionError
=========================== short test summary info ============================
FAILED test_tuple.py::test_mean_of_two_readings - AssertionError: the mean of...
============================== 1 failed ===============================


Now the report shows that the mean was 5.0, and the expected value in the test is the thing to
correct.

### PytestReturnNotNoneWarning: Test functions should return None


In [40]:
%%writefile scratch/stations/test_return.py
from readings import to_fahrenheit


def test_boiling_point_in_fahrenheit():
    return to_fahrenheit(100) == 213


Writing scratch/stations/test_return.py


In [41]:
run_pytest("test_return.py")


============================= test session starts ==============================
collected 1 item

test_return.py .                                                         [100%]

=============================== warnings summary ===============================
test_return.py::test_boiling_point_in_fahrenheit
  _pytest/python.py:161: PytestReturnNotNoneWarning: Test functions should return None, but test_return.py::test_boiling_point_in_fahrenheit returned <class 'bool'>.
  Did you mean to use `assert` instead of `return`?
  See https://docs.pytest.org/en/stable/how-to/assert.html#return-not-none for more information.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
========================= 1 passed, 1 warning =========================


`return` hands the result of the comparison back to pytest, which does nothing with it, so the test
passed while its comparison was `False`. pytest 8.4 warns and counts the test as passed. A test
checks with `assert`, and its expected value comes from outside the code, as the **Why Test**
notebook said: water boils at 212 °F.


In [42]:
%%writefile scratch/stations/test_return.py
from readings import to_fahrenheit


def test_boiling_point_in_fahrenheit():
    assert to_fahrenheit(100) == 212


Overwriting scratch/stations/test_return.py


In [43]:
run_pytest("test_return.py")


============================= test session starts ==============================
collected 1 item

test_return.py .                                                         [100%]

============================== 1 passed ===============================


### ModuleNotFoundError: No module named 'reading'


In [44]:
%%writefile scratch/stations/test_import.py
from reading import mean


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


Writing scratch/stations/test_import.py


In [45]:
report = pytest_report("test_import.py", "-q")
print("\n".join(line for line in report.splitlines() if line.startswith(("E ", "ERROR", "!!!", "1 error"))))


E   ModuleNotFoundError: No module named 'reading'
ERROR test_import.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error


An error, not a failure: pytest could not import the test file, so it collected nothing from it and
stopped the run, with exit code 2. The whole report also shows the traceback through Python's own
import machinery, whose file paths differ from one computer to another, so the cell printed the lines
that matter: the `E` line with the exception, and the counts. `reading` is a typo for `readings`:


In [46]:
%%writefile scratch/stations/test_import.py
from readings import mean


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


Overwriting scratch/stations/test_import.py


In [47]:
run_pytest("test_import.py", "-q")


.                                                                        [100%]
1 passed


### assert 0.15000000000000002 == 0.15


In [48]:
%%writefile scratch/stations/test_floats.py
from readings import mean


def test_mean_of_two_small_readings():
    assert mean([0.1, 0.2]) == 0.15


Writing scratch/stations/test_floats.py


In [49]:
run_pytest("test_floats.py")


============================= test session starts ==============================
collected 1 item

test_floats.py F                                                         [100%]

=================================== FAILURES ===================================
_______________________ test_mean_of_two_small_readings ________________________

    def test_mean_of_two_small_readings():
>       assert mean([0.1, 0.2]) == 0.15
E       assert 0.15000000000000002 == 0.15
E        +  where 0.15000000000000002 = mean([0.1, 0.2])

test_floats.py:5: AssertionError
=========================== short test summary info ============================
FAILED test_floats.py::test_mean_of_two_small_readings - assert 0.15000000000...
============================== 1 failed ===============================


Neither 0.1 nor 0.2 is exact in binary, so their mean is a whisker above 0.15, and `==` compares
every bit. `pytest.approx` compares within a tolerance, which by default is a millionth of the
expected value:


In [50]:
%%writefile scratch/stations/test_floats.py
import pytest

from readings import mean


def test_mean_of_two_small_readings():
    assert mean([0.1, 0.2]) == pytest.approx(0.15)


Overwriting scratch/stations/test_floats.py


In [51]:
run_pytest("test_floats.py")


============================= test session starts ==============================
collected 1 item

test_floats.py .                                                         [100%]

============================== 1 passed ===============================


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
module, every test file, and pytest's cache:


In [52]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- pytest collects tests by name: from files named `test_*.py` or `*_test.py`, the functions whose
  names start with `test`, and those methods of classes whose names start with `Test`.
- `python -m pytest` runs them in a new Python, and reports `.` for a pass and `F` for a failure, the
  details of every failure, and the counts.
- A failure's details show the line that raised and, for a plain `assert`, the values:
  `assert 57.6 == 32`, `where 57.6 = to_fahrenheit(0)`.
- Run part of a suite by file, by node ID (`test_readings.py::test_mean_of_two_readings`), or by
  keyword with `-k`. `-x` stops at the first failure, and `--lf` runs only the failures again.
- The exit code is 0 when every test passed, 1 when a test failed, 2 when a test file could not be
  imported, and 5 when nothing was collected.
- A file or a function with the wrong name is skipped without a word, and `--collect-only` shows
  what pytest found.
- Compare floats with `pytest.approx`, and never wrap an `assert` and its message in parentheses.


## What is next

The **Test Structure** notebook looks inside a test: the steps it takes, arrange, act and assert, a
name that says what it checks, one behavior to a test, and where a project keeps its test files.


---

&#8592; **Previous:** [Why Test](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/01-why-test.ipynb)  &nbsp;·&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)  &nbsp;·&nbsp;  **Next:** [Test Structure](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/03-test-structure.ipynb) &#8594;
